<a href="https://colab.research.google.com/github/Ziyi-star/Bachelorarbeit/blob/main/notebooks/training/_train_simclr_har_0.5s_hyperparameter_feintuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hyperparameter Tuning for SimCLR Classification with Keras Tuner


In [2]:
!pip install keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.1/129.1 kB 5.1 MB/s eta 0:00:00


In [ ]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
# For Vscode
import keras_tuner as kt
import tensorflow as tf
import datetime

import os
import pickle
import scipy
import datetime
import numpy as np
import tensorflow as tf
import simclr_utitlities
import transformations
import simclr_models
import sys
import matplotlib.pyplot as plt

sys.path.append('../')   # Add parent directory to Python path
working_directory = "../models/"
with open('../data_all/TrainTest/X_train_normalized.pkl', 'rb') as f:
    np_train_data = pickle.load(f)
with open('../data_all/TrainTest/y_train_onehot.pkl', 'rb') as f:
    np_train_labels = pickle.load(f)
with open('../data_all/TrainTest/X_val_normalized.pkl', 'rb') as f:
    np_val_data = pickle.load(f)
with open('../data_all/TrainTest/y_val_onehot.pkl', 'rb') as f:
    np_val_labels = pickle.load(f)
with open('../data_all/TrainTest/X_test_data.pkl', 'rb') as f:
    np_test_data = pickle.load(f)
with open('../data_all/TrainTest/y_test_onehot.pkl', 'rb') as f:
    np_test_labels = pickle.load(f)

In [3]:
# For Google Colab
# 1. Clone your repository to go to access your notebook and .py files
!git clone https://github.com/Ziyi-star/Bachelorarbeit.git
# 2. Change working directory to where your notebook and .py files are
import os
os.chdir('/content/Bachelorarbeit/notebooks/training')
import os
import pickle
import scipy
import datetime
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import simclr_utitlities
import transformations
import simclr_models
import sys

import keras_tuner as kt
import tensorflow as tf
import datetime

seed = 1
tf.random.set_seed(seed)
np.random.seed(seed)

# Load data
# Download the file from GitHub
!rm -f *.pkl # Remove existing pickle files
!wget https://raw.githubusercontent.com/Ziyi-star/Bachelorarbeit/main/notebooks/data_all/TrainTest/X_train_normalized.pkl
!wget https://raw.githubusercontent.com/Ziyi-star/Bachelorarbeit/main/notebooks/data_all/TrainTest/y_train_onehot.pkl


working_directory = "../models/"  # Relative path to models folder

# Load as usual
import pickle

with open('X_train_normalized.pkl', 'rb') as f:
    np_train_data = pickle.load(f)
with open('y_train_onehot.pkl', 'rb') as f:
    np_train_labels = pickle.load(f)

print(np_train_data.shape, np_train_labels.shape)

Cloning into 'Bachelorarbeit'...
remote: Enumerating objects: 482, done.
remote: Counting objects: 100% (83/83), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 482 (delta 47), reused 59 (delta 28), pack-reused 399 (from 2)
Receiving objects: 100% (482/482), 147.16 MiB | 27.36 MiB/s, done.
Resolving deltas: 100% (272/272), done.
Updating files: 100% (72/72), done.
--2025-09-22 17:58:11--  https://raw.githubusercontent.com/Ziyi-star/Bachelorarbeit/main/notebooks/data_all/TrainTest/X_train_normalized.pkl
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6304965 (6.0M) [application/octet-stream]
Saving to: ‘X_train_normalized.pkl’

X_train_normalized. 100%[===================>]   6.01M  --.-KB/s    in 0.05s   

2025-09-22 17:58:11 (113 MB/s) 

In [3]:
# Parameters for all experiments
window_size = 50
input_shape = (window_size, 3)
# Time
start_time = datetime.datetime.now()
start_time_str = start_time.strftime("%Y%m%d-%H%M%S")
#Formats the date and time as a string like 20250827-153045 (for filenames, logs, etc.).
tf.keras.backend.set_floatx('float32')
#Sets TensorFlow to use 32-bit floating point numbers for all computations (standard for deep learning).


In [4]:
#A parameter for the SimCLR loss function that controls how sharply similarities are measured.
transform_funcs = [
    transformations.time_segment_permutation_transform_improved,
    transformations.channel_shuffle_transform_vectorized
]
#List of data augmentation functions to apply to the input data. Here, only rotation is used.
transformation_function = simclr_utitlities.generate_composite_transform_function_simple(transform_funcs)

0 <function time_segment_permutation_transform_improved at 0x000001B89D24BE20>
1 <function channel_shuffle_transform_vectorized at 0x000001B89D24BD80>


## 1. Define the Model Building Function for Keras Tuner

In [ ]:
def build_tunable_model(hp):
    """Build a tunable fine-tuning model with hyperparameters."""
    # Load pre-trained SimCLR model
    simclr_model = tf.keras.models.load_model(simclr_model_save_path)
    
    # Tunable parameters
    intermediate_layer = hp.Int('intermediate_layer', min_value=5, max_value=9, step=1)
    last_freeze_layer = hp.Int('last_freeze_layer', min_value=0, max_value=6, step=2)
    
    # Dense layer units
    dense_units = hp.Int('dense_units', min_value=256, max_value=1024, step=256)
    
    # Learning rate
    learning_rate = hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='log')
    
    # Get output from intermediate layer
    intermediate_x = simclr_model.layers[intermediate_layer].output
    
    # Build classification head
    x = tf.keras.layers.Dense(dense_units, activation='relu')(intermediate_x)
    x = tf.keras.layers.Dense(output_shape)(x)
    outputs = tf.keras.layers.Softmax()(x)
    
    # Create model
    model = tf.keras.Model(inputs=simclr_model.inputs, outputs=outputs, name="TPN_tuned")
    
    # Set trainable and non-trainable layers
    for layer in model.layers:
        layer.trainable = False
    
    for layer in model.layers[last_freeze_layer+1:]:
        layer.trainable = True
    
    # Compile model
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss=tf.keras.losses.CategoricalCrossentropy(),
        metrics=[
            tf.keras.metrics.CategoricalAccuracy(name="categorical_accuracy"),
            tf.keras.metrics.AUC(name="auc"),
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall")
        ]
    )
    
    return model

## 2.Set Up and Run the Hyperparameter Tuning

In [ ]:
# Define the batch sizes to try
batch_sizes = [64, 128, 256, 512]

# Create a dictionary to store results for different batch sizes
batch_size_results = {}

# Iterate through batch sizes
for batch_size in batch_sizes:
    print(f"\n\nTesting batch size: {batch_size} ===\n")
    
    # Define the hyperparameter search for this batch size
    tuner = kt.Hyperband(
        build_tunable_model,
        objective='val_categorical_accuracy',
        max_epochs=50,
        factor=3,
        directory=f'keras_tuner/batch_size_{batch_size}',
        project_name=f'simclr_finetuning_{start_time_str}'
    )
    
    # Set up early stopping to prevent overfitting during tuning
    stop_early = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10)
    
    # Start the hyperparameter search with current batch size
    tuner.search(
        x=np_train_data,
        y=np_train_labels,
        epochs=50,  # Reduced for faster iterations across batch sizes
        batch_size=batch_size,
        callbacks=[stop_early],
        validation_data=(np_val_data, np_val_labels)
    )
    
    # Get the best hyperparameters for this batch size
    best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
    
    # Build the model with the best hyperparameters
    best_model = tuner.hypermodel.build(best_hps)
    
    # Evaluate the model to get validation accuracy
    val_loss, val_acc, val_auc, val_precision, val_recall = best_model.evaluate(
        np_val_data, np_val_labels, verbose=0
    )
    
    # Store results for this batch size
    batch_size_results[batch_size] = {
        'hyperparameters': best_hps,
        'val_accuracy': val_acc,
        'val_loss': val_loss
    }
    
    print(f"Batch Size: {batch_size}")
    print(f"Best intermediate layer: {best_hps.get('intermediate_layer')}")
    print(f"Best last freeze layer: {best_hps.get('last_freeze_layer')}")
    print(f"Best dense units: {best_hps.get('dense_units')}")
    print(f"Best learning rate: {best_hps.get('learning_rate')}")
    print(f"Validation Accuracy: {val_acc:.4f}")
    print(f"Validation Loss: {val_loss:.4f}")

# Find the best batch size
best_batch_size = max(batch_size_results.keys(), key=lambda b: batch_size_results[b]['val_accuracy'])
best_overall_hps = batch_size_results[best_batch_size]['hyperparameters']

print("\n===== Best Overall Configuration =====")
print(f"Best Batch Size: {best_batch_size}")
print(f"Best intermediate layer: {best_overall_hps.get('intermediate_layer')}")
print(f"Best last freeze layer: {best_overall_hps.get('last_freeze_layer')}")
print(f"Best dense units: {best_overall_hps.get('dense_units')}")
print(f"Best learning rate: {best_overall_hps.get('learning_rate')}")
print(f"Validation Accuracy: {batch_size_results[best_batch_size]['val_accuracy']:.4f}")